In [1]:
import pandas as pd
from pathlib import Path
from bs4 import BeautifulSoup
import dateparser

In [2]:
BASE_PATH = "../database/Nacarelli/Takeout/YouTube e YouTube Music"

In [3]:
def gerar_dataset_modelo(base_path):
    base_dir = Path(base_path) / "histórico"
    arquivo = base_dir / "histórico-de-visualização.html"

    with open(arquivo, "r", encoding="utf-8") as f:
        soup = BeautifulSoup(f, "html.parser")

    dados = []

    blocos = soup.find_all(
        "div",
        class_="content-cell mdl-cell mdl-cell--6-col mdl-typography--body-1"
    )

    for bloco in blocos:
        link_tag = bloco.find("a")
        if not link_tag:
            continue

        url = link_tag["href"]  # ✅ pega o link
        texto = bloco.get_text(separator=" ", strip=True)
        tipo = texto.split(link_tag.text.strip())[0].strip()

        if "watched" not in tipo.lower():  # só vídeos assistidos
            continue

        dados.append([url])  # só adiciona o link

    df = pd.DataFrame(dados, columns=["titulo"])
    df["proximo_video"] = df["titulo"].shift(-1)  # próximo vídeo também será link

    df = df.dropna()
    df.to_csv("df_modelo_links.csv", index=False)

    return df

markov_df = gerar_dataset_modelo(BASE_PATH)
markov_df.head(20)

,titulo,proximo_video
0,https://music.youtube.com/watch?v=Uyka5SnxmQ4,https://music.youtube.com/watch?v=7BW_c2-TyhA
1,https://music.youtube.com/watch?v=7BW_c2-TyhA,https://music.youtube.com/watch?v=p3eExLblqZQ
2,https://music.youtube.com/watch?v=p3eExLblqZQ,https://music.youtube.com/watch?v=OM6KZ043DAQ
3,https://music.youtube.com/watch?v=OM6KZ043DAQ,https://music.youtube.com/watch?v=D8RZbI1jfQ0
4,https://music.youtube.com/watch?v=D8RZbI1jfQ0,https://music.youtube.com/watch?v=wlzGXcTzdzU
5,https://music.youtube.com/watch?v=wlzGXcTzdzU,https://music.youtube.com/watch?v=rC-bvpmyYsQ
6,https://music.youtube.com/watch?v=rC-bvpmyYsQ,https://music.youtube.com/watch?v=h27qhWrgXIg
7,https://music.youtube.com/watch?v=h27qhWrgXIg,https://music.youtube.com/watch?v=Pr-R3ov6OZc
8,https://music.youtube.com/watch?v=Pr-R3ov6OZc,https://music.youtube.com/watch?v=lMMGvFS_KqE
9,https://music.youtube.com/watch?v=lMMGvFS_KqE,https://music.youtube.com/watch?v=vum2bUJwfkI


In [4]:
# ler CSV
df_modelo = pd.read_csv("markov_transicoes.csv")

# filtro para links do YouTube (tanto título quanto próximo vídeo)
df_modelo_links = df_modelo[
    df_modelo["titulo"].str.contains(r"^https://www\.youtube\.com/watch\?v=", na=False) &
    df_modelo["proximo_video"].str.contains(r"^https://www\.youtube\.com/watch\?v=", na=False)
]

# verificação
print(df_modelo_links.shape)
print(df_modelo_links.head())

# salvar CSV filtrado
df_modelo_links.to_csv("df_modelo_links.csv", index=False)

(20, 4)
                                          titulo  \
257  https://www.youtube.com/watch?v=-nfbQvU1PnQ   
263  https://www.youtube.com/watch?v=6cHQj9TJlko   
264  https://www.youtube.com/watch?v=79DwTNO9P_s   
270  https://www.youtube.com/watch?v=BtIrAwcbqtQ   
271  https://www.youtube.com/watch?v=DbHssEeKEFs   

                                   proximo_video  contagem  probabilidade  
257  https://www.youtube.com/watch?v=rL1jI4k-VSE         1            1.0  
263  https://www.youtube.com/watch?v=2th7dPiXSpY         1            1.0  
264  https://www.youtube.com/watch?v=ayajTrETG48         1            1.0  
270  https://www.youtube.com/watch?v=KQO0po1LmXc         1            1.0  
271  https://www.youtube.com/watch?v=MTOx12b8WWQ         1            1.0  
